In [ ]:
import random, math, csv
import numpy as np
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet50
from tqdm.auto import tqdm
import plotly.graph_objects as go

# ============================================================
# CONFIG
# ============================================================
SEED        = 41
EPOCHS      = 150          # a bit longer to improve generalization
BATCH_SIZE  = 128

LR           = 0.05        # CIFAR-100 learning rate for the adapted ResNet-50
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4

ALPHA_CE     = 1.0        # CE weight
HINGE_WEIGHT = 2.0        # margin hinge weight (slightly stronger than 1.0)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
PERSISTENT  = False
PIN_MEMORY  = (DEVICE == "cuda")

MARGIN        = 1.0
SIGMA_Q       = 0.4       # example shift level for printing
ALPHA         = 0.2       # nominal miscoverage (target coverage 1-ALPHA)
CLS_SCALE     = 1.0       # global scale on logits (effective Lipschitz)

# σ-grid for coverage / set-size surfaces
SIGMA_MIN_GRID = 0.0
SIGMA_MAX_GRID = 1.5
N_SIGMA_GRID   = 11
SIGMA_GRID     = np.linspace(SIGMA_MIN_GRID, SIGMA_MAX_GRID, N_SIGMA_GRID)

NUM_CLASSES = 100

# Smoothing for plots (moving average along epochs)
SMOOTH_WINDOW_EPOCHS = 3  # set to 1 to disable smoothing

# ============================================================
# SEED
# ============================================================
def set_seed(seed: int = 43):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ============================================================
# MODEL: ResNet-50 feature extractor + linear head
# ============================================================
class ResNetCIFAR100_FeatureHead(nn.Module):
    """
    ResNet-18 backbone adapted for 32x32 CIFAR:
      - conv1: 3x3, stride=1, padding=1
      - maxpool removed
      - backbone.fc replaced by Identity -> outputs latent features φ(x)
      - separate linear classifier on top of φ(x)
    """
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        backbone = resnet50(weights=None)
        # Adapt first conv + remove maxpool for CIFAR
        backbone.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        backbone.maxpool = nn.Identity()

        # Make backbone output features (after avgpool + flatten)
        num_feats = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        # linear head (classifier = latent -> logits)
        self.classifier = nn.Linear(num_feats, num_classes)

    def forward(self, x):
        # φ(x)
        feats = self.backbone(x)         # [B, D_latent]
        logits = self.classifier(feats)  # [B, K]
        return CLS_SCALE * logits

    @torch.no_grad()
    def logits_from_x_nograd(self, x):
        feats = self.backbone(x)
        logits = self.classifier(feats)
        return CLS_SCALE * logits

    @torch.no_grad()
    def features_from_x_nograd(self, x):
        """
        Latent features φ(x) in the space where:
          - ρ_mix(σ) is measured as ||φ(x) - φ(T_σ(x))||_2
          - Lipschitz L_γ is computed for the linear head on top of these features.
        """
        return self.backbone(x)


# ============================================================
# LOSSES / METRICS
# ============================================================
def cross_entropy_from_logits(logits, y, label_smoothing: float = 0.0):
    # you can set label_smoothing>0 if you like; 0.0 keeps CE sharp
    return F.cross_entropy(logits, y, reduction='mean', label_smoothing=label_smoothing)

def margin_hinge_loss_from_logits(logits, targets, margin=1.0):
    """
    Hinge loss with margin parameter 'margin':
      h(x,y) = max(0, margin - (logit_y - max_{k≠y} logit_k)).
    """
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    v = F.relu(margin - m)
    return v.mean()

@torch.no_grad()
def samplewise_margin_and_hinge(logits, targets, margin=1.0):
    """
    Per-sample margin and hinge (no averaging).
    """
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    h = F.relu(margin - m)
    return m, h


# ============================================================
# DATA & LOADERS (CIFAR-100)
# ============================================================
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

transform_test  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

train_set = datasets.CIFAR100(root='./data', train=True,  download=True, transform=transform_train)
test_set  = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

def make_half_splits(dataset, batch_size=128):
    """
    Split 'dataset' into calibration and test halves,
    returning non-shuffled loaders (for conformal purposes).
    """
    n = len(dataset)
    idx_cal = np.arange(0, n//2)
    idx_tst = np.arange(n//2, n)
    ds_cal  = Subset(dataset, idx_cal)
    ds_tst  = Subset(dataset, idx_tst)
    ld_cal  = DataLoader(ds_cal, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    ld_tst  = DataLoader(ds_tst, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    return ld_cal, ld_tst

# P = source (unshifted test distribution)
P_cal_loader, P_tst_loader = make_half_splits(test_set, BATCH_SIZE)
# Q realized by T_sigma at scoring time
Q_cal_loader, Q_tst_loader = P_cal_loader, P_tst_loader

# ============================================================
# Q-SHIFT TRANSFORM T_sigma
# ============================================================
def apply_Q_transform(x: torch.Tensor, sigma: float) -> torch.Tensor:
    """
    Structured shift T_sigma(x) used to define Q for CIFAR-100:
      - global per-image brightness/contrast jitter
      - small integer translations (torch.roll)
      - mild additive Gaussian noise
    sigma in [0, 1], sigma=0 => identity.
    """
    if sigma <= 0.0:
        return x

    B = x.size(0)
    device = x.device

    # brightness/contrast jitter
    contrast_scale = 1.0 + 0.8 * sigma * (2.0 * torch.rand(B, 3, 1, 1, device=device) - 1.0)
    brightness_shift = 0.4 * sigma * (2.0 * torch.rand(B, 3, 1, 1, device=device) - 1.0)
    x = x * contrast_scale + brightness_shift

    # translations
    max_shift = int(4 * sigma)
    if max_shift > 0:
        shift_x = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        shift_y = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        for i in range(B):
            x[i] = torch.roll(x[i], shifts=(int(shift_y[i]), int(shift_x[i])), dims=(1, 2))

    # Gaussian noise
    noise_std = 0.1 * sigma
    if noise_std > 0:
        x = x + noise_std * torch.randn_like(x)

    return x

# ============================================================
# MODEL + OPTIMIZER + SCHEDULER
# ============================================================
model = ResNetCIFAR100_FeatureHead(num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

# Step-down LR schedule (standard-ish)
milestone1 = int(EPOCHS * 0.5)   # e.g. epoch 30 if EPOCHS=60
milestone2 = int(EPOCHS * 0.75)  # e.g. epoch 45
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[milestone1, milestone2],
    gamma=0.1,
)

# ============================================================
# TRAIN / EVAL
# ============================================================
def train_one_epoch(epoch: int):
    model.train()
    sums = {'cce':0.0,'hinge':0.0,'acc':0.0,'n':0}
    pbar = tqdm(DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT),
                desc=f"Train {epoch:02d}")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        cce   = cross_entropy_from_logits(logits, y, label_smoothing=0.0)
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN)
        loss  = ALPHA_CE * cce + HINGE_WEIGHT * hinge

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            acc = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        sums['cce']   += cce.item()*bs
        sums['hinge'] += hinge.item()*bs
        sums['acc']   += acc*bs
        sums['n']     += bs
        pbar.set_postfix({k:f"{sums[k]/sums['n']:.4f}" for k in ['cce','hinge','acc']})
    for k in ['cce','hinge','acc']:
        sums[k] /= sums['n']
    return sums

@torch.no_grad()
def evaluate(loader, tag="Eval"):
    model.eval()
    tot = {'cce':0.0,'hinge':0.0,'acc':0.0,'n':0}
    pbar = tqdm(loader, desc=f"{tag}", leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        cce   = F.cross_entropy(logits, y, reduction='mean').item()
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN).item()
        acc   = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        tot['cce']  += cce*bs
        tot['hinge']+= hinge*bs
        tot['acc']  += acc*bs
        tot['n']    += bs
    for k in ['cce','hinge','acc']:
        tot[k] /= tot['n']
    return tot

# ============================================================
# CONFORMAL HELPERS
# ============================================================
@torch.no_grad()
def predictive_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    probs = F.softmax(logits, dim=1)
    log_probs = (probs + 1e-12).log()
    ent = -(probs * log_probs).sum(dim=1)
    return ent  # [B]

@torch.no_grad()
def scores_for_labels_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> np.ndarray:
    B = logits.size(0)
    idx = torch.arange(B, device=logits.device)
    masked = logits.clone()
    masked[idx, labels] = -float('inf')
    max_other = masked.max(dim=1).values
    s = (max_other - logits[idx, labels]).detach().float().cpu().numpy()
    return s

def conformal_quantile(scores: np.ndarray, alpha: float, finite_sample: bool = True) -> float:
    n = scores.size
    if n == 0:
        return float('nan')
    if finite_sample:
        qlevel = math.ceil((n + 1) * (1 - alpha)) / n
    else:
        qlevel = 1 - alpha
    try:
        return float(np.quantile(scores, qlevel, method='higher'))
    except TypeError:
        return float(np.quantile(scores, qlevel, interpolation='higher'))

def coverage_given_threshold(scores_true: np.ndarray, threshold: float) -> float:
    if scores_true.size == 0:
        return float('nan')
    return float((scores_true <= threshold).mean())

# -----------------------------
# Collect true-label scores on P or Q
# -----------------------------
@torch.no_grad()
def collect_true_scores(loader: DataLoader, sigma: float) -> np.ndarray:
    model.eval()
    all_scores = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        s = scores_for_labels_from_logits(logits, y)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

# -----------------------------
# Expected set size for a given threshold q
# -----------------------------
@torch.no_grad()
def expected_set_size_threshold(loader: DataLoader, sigma: float, q: float) -> float:
    """
    For each x, prediction set:
      C_q(x) = { y : s(x,y) <= q },
    where s(x,y) = max_{k≠y} logit_k - logit_y.
    Return average |C_q(x)|.
    """
    model.eval()
    total_size = 0.0
    total_n = 0

    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)  # [B,K]
        B, K = logits.shape

        values, indices = logits.topk(k=2, dim=1)  # [B,2]
        max1 = values[:, 0:1]
        max2 = values[:, 1:2]
        argmax = indices[:, 0]

        classes = torch.arange(K, device=logits.device).unsqueeze(0).expand(B, -1)
        argmax_exp = argmax.unsqueeze(1).expand(-1, K)
        max_other = torch.where(classes == argmax_exp, max2, max1)

        scores_all = max_other - logits
        set_sizes = (scores_all <= q).sum(dim=1).float()

        total_size += set_sizes.sum().item()
        total_n += B

    return total_size / total_n if total_n > 0 else float('nan')

# -----------------------------
# Mixed pseudo-label scores on P or Q
# -----------------------------
@torch.no_grad()
def collect_pseudo_scores_mixed(
    loader: DataLoader,
    sigma: float,
    u: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    """
    Mixed pseudo-labeling:
      If ent(x) <= u: pseudo-label = argmax_k logit_k
      else:          pseudo-label ~ Uniform([K])
    """
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        yhat = logits.argmax(dim=1)

        B, K = logits.shape
        rand_labels = torch.randint(low=0, high=num_classes, size=(B,), device=logits.device)
        mask_low = (ent <= u)
        pseudo_y = torch.where(mask_low, yhat, rand_labels)

        s = scores_for_labels_from_logits(logits, pseudo_y)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

# -----------------------------
# Compute u_P^* on P
# -----------------------------
@torch.no_grad()
def compute_uP_star(
    P_cal_loader: DataLoader,
    P_tst_loader: DataLoader,
    alpha: float = ALPHA,
    num_u_points: int = 30,
):
    """
    Approximate u_P^* by scanning u on entropy quantiles of P_cal.
    """
    # entropies on P_cal
    ent_all = []
    for x, _ in P_cal_loader:
        x = x.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        ent_all.append(ent.cpu().numpy())
    ent_all = np.concatenate(ent_all, axis=0)

    low_q, high_q = 0.05, 0.95
    t_min = np.quantile(ent_all, low_q)
    t_max = np.quantile(ent_all, high_q)
    u_cutoffs = np.linspace(t_min, t_max, num_u_points)

    scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)
    cov_P_vals = []
    target_cov = 1.0 - alpha

    u_P_star = u_cutoffs[0]
    q_Phat_uP_star = None

    for u in u_cutoffs:
        scores_P_cal_pseudo = collect_pseudo_scores_mixed(
            loader=P_cal_loader,
            sigma=0.0,
            u=u,
            num_classes=NUM_CLASSES,
        )
        q_Phat_u = conformal_quantile(scores_P_cal_pseudo, alpha, finite_sample=True)
        cov_P_u = coverage_given_threshold(scores_P_tst_true, q_Phat_u)
        cov_P_vals.append(cov_P_u)

        # Track largest u with coverage >= target
        if cov_P_u >= target_cov:
            u_P_star = u
            q_Phat_uP_star = q_Phat_u

    cov_P_vals = np.array(cov_P_vals)

    # If never feasible, fallback to last u
    if q_Phat_uP_star is None:
        scores_P_cal_pseudo = collect_pseudo_scores_mixed(
            loader=P_cal_loader,
            sigma=0.0,
            u=u_cutoffs[-1],
            num_classes=NUM_CLASSES,
        )
        q_Phat_uP_star = conformal_quantile(scores_P_cal_pseudo, alpha, finite_sample=True)
        u_P_star = float(u_cutoffs[-1])

    return float(u_P_star), u_cutoffs, cov_P_vals, float(q_Phat_uP_star)

# ============================================================
# STORAGE FOR PER-EPOCH STATS
# ============================================================
hinge_P_epochs         = np.zeros(EPOCHS, dtype=np.float64)                  # hinge loss on P_tst
cov_Q_uP_epochs_sigma  = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)  # coverage on Q per σ
cov_P_uP_epochs        = np.zeros(EPOCHS, dtype=np.float64)                  # coverage on P using u_P^*
u_P_star_epochs        = np.zeros(EPOCHS, dtype=np.float64)                  # u_P^* per epoch

ESS_P_uP_epochs        = np.zeros(EPOCHS, dtype=np.float64)
ESS_P_oracle_epochs    = np.zeros(EPOCHS, dtype=np.float64)
ESS_Q_uP_epochs_sigma      = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)
ESS_Q_oracle_epochs_sigma  = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)

# ============================================================
# MAIN TRAINING + COVERAGE / SET-SIZE LOOP
# ============================================================
print("=== Training with per-epoch source-tuned pseudo-label coverage & set-size tracking (ResNet-50 feature+head, CIFAR-100) ===")

for epoch in range(1, EPOCHS+1):
    # 1) Train one epoch
    tr = train_one_epoch(epoch)

    # 2) Evaluate on P_tst to get hinge loss on P
    teP = evaluate(P_tst_loader, tag=f"Test(P) epoch {epoch}")
    hinge_P = teP["hinge"]
    hinge_P_epochs[epoch-1] = hinge_P

    # 3) Compute u_P^* for current model (source-tuned)
    u_P_star, u_grid, cov_P_over_u, q_Phat_uP_star = compute_uP_star(
        P_cal_loader=P_cal_loader,
        P_tst_loader=P_tst_loader,
        alpha=ALPHA,
        num_u_points=30,
    )
    u_P_star_epochs[epoch-1] = u_P_star

    # 3a) Coverage on P using u_P^*
    scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)
    cov_P_uP_star = coverage_given_threshold(scores_P_tst_true, q_Phat_uP_star)
    cov_P_uP_epochs[epoch-1] = cov_P_uP_star

    # 3b) Oracle q_P from true labels on P_cal
    scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
    q_P_true_oracle = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

    # Expected set size on P with u_P^* vs oracle
    ESS_P_uP = expected_set_size_threshold(P_tst_loader, sigma=0.0, q=q_Phat_uP_star)
    ESS_P_oracle = expected_set_size_threshold(P_tst_loader, sigma=0.0, q=q_P_true_oracle)
    ESS_P_uP_epochs[epoch-1] = ESS_P_uP
    ESS_P_oracle_epochs[epoch-1] = ESS_P_oracle

    print(
        f"Epoch {epoch:03d} | "
        f"Train CE {tr['cce']:.4f} hinge {tr['hinge']:.4f} acc {tr['acc']:.4f} | "
        f"Test(P) CE {teP['cce']:.4f} hinge {teP['hinge']:.4f} acc {teP['acc']:.4f} | "
        f"u_P^*={u_P_star:.4f}, Cov_P(u_P^*)={cov_P_uP_star:.4f}, "
        f"ESS_P(u_P^*)={ESS_P_uP:.2f}, ESS_P(oracle)={ESS_P_oracle:.2f}"
    )

    # LR schedule step
    scheduler.step()

    # 4) For each sigma, compute coverage and expected set size on Q using u_P^* and oracle q_Q
    for s_idx, sigma in enumerate(SIGMA_GRID):
        sigma_val = float(sigma)

        # pseudo-scores on Q_cal with mixed rule at u_P_star
        scores_Q_cal_pseudo = collect_pseudo_scores_mixed(
            loader=Q_cal_loader,
            sigma=sigma_val,
            u=u_P_star,
            num_classes=NUM_CLASSES,
        )
        q_Qhat_uP_sigma = conformal_quantile(scores_Q_cal_pseudo, ALPHA, finite_sample=True)

        # oracle q_Q from true-label conformal on Q_cal
        scores_Q_cal_true = collect_true_scores(Q_cal_loader, sigma=sigma_val)
        q_Q_true_sigma = conformal_quantile(scores_Q_cal_true, ALPHA, finite_sample=True)

        # true scores on Q_tst (for coverage numbers)
        scores_Q_tst_true = collect_true_scores(
            loader=Q_tst_loader,
            sigma=sigma_val,
        )
        cov_Q_uP = coverage_given_threshold(scores_Q_tst_true, q_Qhat_uP_sigma)
        cov_Q_uP_epochs_sigma[epoch-1, s_idx] = cov_Q_uP

        # Expected set size on Q with u_P^* and oracle q_Q
        ESS_Q_uP = expected_set_size_threshold(Q_tst_loader, sigma=sigma_val, q=q_Qhat_uP_sigma)
        ESS_Q_oracle = expected_set_size_threshold(Q_tst_loader, sigma=sigma_val, q=q_Q_true_sigma)
        ESS_Q_uP_epochs_sigma[epoch-1, s_idx] = ESS_Q_uP
        ESS_Q_oracle_epochs_sigma[epoch-1, s_idx] = ESS_Q_oracle

    # Optional: print coverage and ESS on Q at a particular sigma, e.g. SIGMA_Q
    if SIGMA_MIN_GRID <= SIGMA_Q <= SIGMA_MAX_GRID:
        s_idx_print = int(
            round(
                (SIGMA_Q - SIGMA_MIN_GRID)
                / (SIGMA_MAX_GRID - SIGMA_MIN_GRID)
                * (N_SIGMA_GRID - 1)
            )
        )
        cov_Q_at_SIGMAQ = cov_Q_uP_epochs_sigma[epoch-1, s_idx_print]
        ESS_Q_uP_at_SIGMAQ = ESS_Q_uP_epochs_sigma[epoch-1, s_idx_print]
        ESS_Q_oracle_at_SIGMAQ = ESS_Q_oracle_epochs_sigma[epoch-1, s_idx_print]
        print(
            f"  Q at σ={SIGMA_Q:.2f} using u_P^*: "
            f"Coverage={cov_Q_at_SIGMAQ:.4f}, "
            f"ESS_Q(u_P^*)={ESS_Q_uP_at_SIGMAQ:.2f}, "
            f"ESS_Q(oracle)={ESS_Q_oracle_at_SIGMAQ:.2f}"
        )

# ============================================================
# SMOOTHING HELPERS FOR PLOTS
# ============================================================
def smooth_1d(a: np.ndarray, window: int = 3) -> np.ndarray:
    """Moving-average smoothing with edge padding."""
    a = np.asarray(a, dtype=np.float64)
    if window <= 1 or a.size == 0 or a.size < window:
        return a.copy()
    pad_left = window // 2
    pad_right = window - 1 - pad_left
    a_pad = np.pad(a, (pad_left, pad_right), mode="edge")
    kernel = np.ones(window, dtype=np.float64) / float(window)
    return np.convolve(a_pad, kernel, mode="valid")

def smooth_2d_along_axis0(Z: np.ndarray, window: int = 3) -> np.ndarray:
    """Smooth 2-D array along axis 0 (epochs) using smooth_1d."""
    Z = np.asarray(Z, dtype=np.float64)
    if window <= 1 or Z.size == 0 or Z.shape[0] < window:
        return Z.copy()
    Z_sm = np.empty_like(Z)
    for j in range(Z.shape[1]):
        Z_sm[:, j] = smooth_1d(Z[:, j], window)
    return Z_sm

def smooth_2d_full(Z: np.ndarray, window: int = 3) -> np.ndarray:
    """Smooth 2-D array along epochs and sigma."""
    Z = np.asarray(Z, dtype=np.float64)
    if window <= 1 or Z.size == 0 or min(Z.shape) < window:
        return Z.copy()
    Z1 = smooth_2d_along_axis0(Z, window)
    Z2 = smooth_2d_along_axis0(Z1.T, window).T
    return Z2

def clip_outliers_percentile(Z: np.ndarray, low_q: float = 2.0, high_q: float = 98.0) -> np.ndarray:
    """Clip extreme values of Z to [low_q, high_q] percentiles (for visualization only)."""
    Z = np.asarray(Z, dtype=np.float64)
    Zc = Z.copy()
    finite = np.isfinite(Zc)
    if finite.sum() == 0:
        return Zc
    lo, hi = np.percentile(Zc[finite], [low_q, high_q])
    Zc[Zc < lo] = lo
    Zc[Zc > hi] = hi
    return Zc

# Apply smoothing only for visualization
hinge_P_plot     = smooth_1d(hinge_P_epochs, window=SMOOTH_WINDOW_EPOCHS)
u_P_star_plot    = smooth_1d(u_P_star_epochs, window=SMOOTH_WINDOW_EPOCHS)
cov_Q_uP_plot    = smooth_2d_full(cov_Q_uP_epochs_sigma, window=SMOOTH_WINDOW_EPOCHS)
ESS_Q_uP_plot    = smooth_2d_full(ESS_Q_uP_epochs_sigma, window=SMOOTH_WINDOW_EPOCHS)

print("Smoothing done for plotting (window =", SMOOTH_WINDOW_EPOCHS, ")")

# ------------------------------------------------
# 3D Plot: coverage of u_P^* on Q vs (σ, hinge(P))
# ------------------------------------------------
START_EPOCH_PLOT = 10
start_idx = START_EPOCH_PLOT - 1

LOSS_AXIS  = hinge_P_plot
SIGMA_AXIS = SIGMA_GRID

Z_cov = cov_Q_uP_plot.T
Z_cov = clip_outliers_percentile(Z_cov, low_q=2.0, high_q=98.0)

LOSS_AXIS_sub = LOSS_AXIS[start_idx:]
Z_cov_sub     = Z_cov[:, start_idx:]

order = np.argsort(LOSS_AXIS_sub)
LOSS_AXIS_sorted = LOSS_AXIS_sub[order]
Z_cov_sorted     = Z_cov_sub[:, order]

SIGMA_MESH, LOSS_MESH = np.meshgrid(SIGMA_AXIS, LOSS_AXIS_sorted, indexing="ij")
nominal_cov = 1.0 - ALPHA
Z_nom = nominal_cov * np.ones_like(Z_cov_sorted)

fig3d_cov = go.Figure()
fig3d_cov.add_surface(
    x=SIGMA_MESH,
    y=LOSS_MESH,
    z=Z_cov_sorted,
    colorscale="Viridis",
    opacity=0.9,
    name="Coverage on Q using u_P^*",
    showscale=True,
)
fig3d_cov.add_surface(
    x=SIGMA_MESH,
    y=LOSS_MESH,
    z=Z_nom,
    colorscale=[[0, "lightgrey"], [1, "lightgrey"]],
    opacity=0.3,
    name="Nominal 1−α plane",
    showscale=False,
)
fig3d_cov.update_layout(
    title=(
        "CIFAR-100: Coverage on Q vs (σ, hinge loss on P)\n"
        "Source-tuned mixed pseudo-labeling (u_P^*) with ResNet-18 feature+head"
    ),
    scene=dict(
        xaxis_title="σ (Q shift level)",
        yaxis_title="Hinge loss on P (P_tst)",
        zaxis_title="Coverage on Q",
        zaxis=dict(range=[0.0, 1.02]),
    ),
    width=900,
    height=700,
)
fig3d_cov.write_html("cifar100_resnet_feathead_uP_coverage_vs_sigma_hinge_3d.html",
                     include_plotlyjs="cdn")
print("Saved 3D coverage plot to cifar100_resnet_feathead_uP_coverage_vs_sigma_hinge_3d.html")

# ------------------------------------------------
# 3D Plot: expected set size on Q
# ------------------------------------------------
Z_ess_Q_uP = ESS_Q_uP_plot.T
Z_ess_Q_uP = clip_outliers_percentile(Z_ess_Q_uP, low_q=2.0, high_q=98.0)

Z_ess_Q_uP_sub = Z_ess_Q_uP[:, start_idx:]
Z_ess_Q_uP_sorted = Z_ess_Q_uP_sub[:, order]

fig3d_ess = go.Figure()
fig3d_ess.add_surface(
    x=SIGMA_MESH,
    y=LOSS_MESH,
    z=Z_ess_Q_uP_sorted,
    colorscale="Viridis",
    opacity=0.9,
    name="Expected set size on Q using u_P^*",
    showscale=True,
)
fig3d_ess.update_layout(
    title=(
        "CIFAR-100: Expected set size on Q vs (σ, hinge loss on P)\n"
        "Source-tuned mixed pseudo-labeling (u_P^*) with ResNet-18 feature+head"
    ),
    scene=dict(
        xaxis_title="σ (Q shift level)",
        yaxis_title="Hinge loss on P (P_tst)",
        zaxis_title="Expected set size on Q",
        zaxis=dict(range=[0.5, NUM_CLASSES + 0.5]),
    ),
    width=900,
    height=700,
)
fig3d_ess.write_html("cifar100_resnet_feathead_uP_exp_set_size_vs_sigma_hinge_3d.html",
                     include_plotlyjs="cdn")
print("Saved 3D expected set size plot to cifar100_resnet_feathead_uP_exp_set_size_vs_sigma_hinge_3d.html")

# ------------------------------------------------
# 2D Plot: u_P^* vs hinge loss on P (epochs 10..)
# ------------------------------------------------
import matplotlib.pyplot as plt
import scienceplots
import matplotlib as mpl

plt.style.use(["science", "ieee", "no-latex"])

mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

IEEE_WIDTH_1COL = 5.0
IEEE_ASPECT = 0.9
FIGSIZE_1COL = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

fig_u_vs_hinge = plt.figure(figsize=FIGSIZE_1COL)
ax = fig_u_vs_hinge.add_subplot(111)
ax.plot(
    hinge_P_plot[start_idx:],
    u_P_star_plot[start_idx:],
    marker="o",
    linestyle="-",
    label="u_P^* vs hinge(P_tst)",
)
ax.set_xlabel("Hinge loss on P_tst")
ax.set_ylabel("u_P^* (entropy cutoff)")
ax.grid(True, which="both", linewidth=0.5, alpha=0.5)
ax.legend(frameon=False, loc="best")
plt.tight_layout()
plt.savefig("cifar100_resnet_feathead_uP_vs_hingeP.pdf", bbox_inches="tight")
plt.savefig("cifar100_resnet_feathead_uP_vs_hingeP.png", dpi=300, bbox_inches="tight")
plt.close(fig_u_vs_hinge)
print("Saved u_P^* vs hinge(P) plot to cifar100_resnet_feathead_uP_vs_hingeP.*")

# ------------------------------------------------
# CSV: expected set sizes on P and Q (for all epochs)
# ------------------------------------------------
csv_path = "cifar100_resnet_feathead_uP_expected_set_sizes.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "domain", "sigma", "hinge_P",
                     "ESS_uP", "ESS_oracle"])
    for epoch in range(1, EPOCHS+1):
        hinge = hinge_P_epochs[epoch-1]

        # P row: sigma=0
        writer.writerow([
            epoch,
            "P",
            0.0,
            hinge,
            ESS_P_uP_epochs[epoch-1],
            ESS_P_oracle_epochs[epoch-1],
        ])

        # Q rows
        for s_idx, sigma in enumerate(SIGMA_GRID):
            writer.writerow([
                epoch,
                "Q",
                float(sigma),
                hinge,
                ESS_Q_uP_epochs_sigma[epoch-1, s_idx],
                ESS_Q_oracle_epochs_sigma[epoch-1, s_idx],
            ])

print(f"Saved expected set size table to {csv_path}")

print("\nDone. Arrays available (raw, unsmoothed):")
print("hinge_P_epochs.shape =", hinge_P_epochs.shape)
print("cov_Q_uP_epochs_sigma.shape =", cov_Q_uP_epochs_sigma.shape)
print("cov_P_uP_epochs.shape =", cov_P_uP_epochs.shape)
print("u_P_star_epochs.shape =", u_P_star_epochs.shape)
print("ESS_P_uP_epochs.shape =", ESS_P_uP_epochs.shape)
print("ESS_P_oracle_epochs.shape =", ESS_P_oracle_epochs.shape)
print("ESS_Q_uP_epochs_sigma.shape =", ESS_Q_uP_epochs_sigma.shape)
print("ESS_Q_oracle_epochs_sigma.shape =", ESS_Q_oracle_epochs_sigma.shape)


In [ ]:
import numpy as np
import torch

# -----------------------------
# Helper: hard pseudo-label scores on Q
# -----------------------------
@torch.no_grad()
def collect_pseudo_scores_hard(
    loader: DataLoader,
    sigma: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    """
    Hard pseudo-calibration:
      - apply T_sigma,
      - y_hat = argmax_k logits_k,
      - score s(x, y_hat) = max_{k != y_hat} logit_k - logit_{y_hat}.
    """
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(dim=1)
        s = scores_for_labels_from_logits(logits, yhat)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])


# -----------------------------
# Build table for selected sigmas using final epoch
# -----------------------------
@torch.no_grad()
def make_cifar100_cov_ess_table_latex():
    # Use last epoch (EPOCHS) for the source-tuned stats
    epoch_for_table = EPOCHS          # 1-based
    eidx = epoch_for_table - 1        # 0-based

    # Choose a concise subset of shift levels for the table
    # (edit this list if you want other σ’s)
    TABLE_SIGMAS = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]

    # Source calibration threshold: conformal on P_cal with true labels (σ=0)
    scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
    q_P_source = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

    # Storage for table rows
    rows = []

    for sigma_req in TABLE_SIGMAS:
        # Use nearest σ in the pre-defined SIGMA_GRID for source-tuned stats
        s_idx = int(np.argmin(np.abs(SIGMA_GRID - sigma_req)))
        sigma_used = float(SIGMA_GRID[s_idx])

        # True scores on Q_tst at this sigma (shared across methods)
        scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma_used)

        # 1) Source calibration: threshold fixed from P_cal (σ=0), evaluate on Q_tst(σ)
        cov_source = coverage_given_threshold(scores_Q_tst_true, q_P_source)
        ess_source = expected_set_size_threshold(Q_tst_loader, sigma_used, q_P_source)

        # 2) Hard pseudo calibration on Q
        scores_Q_cal_pseudo_hard = collect_pseudo_scores_hard(
            loader=Q_cal_loader,
            sigma=sigma_used,
            num_classes=NUM_CLASSES,
        )
        q_Qhat_hard = conformal_quantile(scores_Q_cal_pseudo_hard, ALPHA, finite_sample=True)
        cov_hard = coverage_given_threshold(scores_Q_tst_true, q_Qhat_hard)
        ess_hard = expected_set_size_threshold(Q_tst_loader, sigma_used, q_Qhat_hard)

        # 3) Source-tuned pseudo calibration (u_P^*), already tracked during training
        cov_src_tuned = float(cov_Q_uP_epochs_sigma[eidx, s_idx])
        ess_src_tuned = float(ESS_Q_uP_epochs_sigma[eidx, s_idx])

        # 4) Target calibration (ideal): true-label conformal on Q_cal(σ), evaluate on Q_tst(σ)
        scores_Q_cal_true = collect_true_scores(Q_cal_loader, sigma_used)
        q_Q_true = conformal_quantile(scores_Q_cal_true, ALPHA, finite_sample=True)
        cov_target = coverage_given_threshold(scores_Q_tst_true, q_Q_true)
        ess_target = expected_set_size_threshold(Q_tst_loader, sigma_used, q_Q_true)

        rows.append({
            "sigma":    sigma_used,
            "cov_src":  cov_source,
            "ess_src":  ess_source,
            "cov_hard": cov_hard,
            "ess_hard": ess_hard,
            "cov_tuned": cov_src_tuned,
            "ess_tuned": ess_src_tuned,
            "cov_tgt":  cov_target,
            "ess_tgt":  ess_target,
        })

    # -----------------------------
    # Build LaTeX table string
    # -----------------------------
    nominal_cov = 1.0 - ALPHA

    header = r"""\begin{table}[t]
\centering
\caption{CIFAR100: Coverage and expected set size vs.\ shift level $\sigma$ (nominal $1-\alpha=%.2f$).}
\label{tab:cifar100_cov_ess}
\begin{tabular}{c|cc|cc|cc|cc}
\hline
 & \multicolumn{2}{c|}{Source cal} & \multicolumn{2}{c|}{Hard pseudo cal} & \multicolumn{2}{c|}{Source-tuned cal} & \multicolumn{2}{c}{Target cal (ideal)} \\
$\sigma$ & Cov & ESS & Cov & ESS & Cov & ESS & Cov & ESS \\
\hline
""" % nominal_cov

    body_lines = []
    for r in rows:
        line = (
            f"{r['sigma']:.2f} & "
            f"{100.0*r['cov_src']:.2f} & {r['ess_src']:.2f} & "
            f"{100.0*r['cov_hard']:.2f} & {r['ess_hard']:.2f} & "
            f"{100.0*r['cov_tuned']:.2f} & {r['ess_tuned']:.2f} & "
            f"{100.0*r['cov_tgt']:.2f} & {r['ess_tgt']:.2f} \\\\"
        )
        body_lines.append(line)

    footer = r"""
\hline
\end{tabular}
\end{table}
"""

    table_latex = header + "\n".join(body_lines) + footer
    print(table_latex)


# Run this after training is done to print the LaTeX table
make_cifar100_cov_ess_table_latex()


In [ ]:
# ============================================================
# Coverage vs σ + ramp-loss & margin-tail lower bounds
#   Empirical coverage:
#       - Source calibration
#       - Hard pseudo-calibration
#       - Source-tuned pseudo-calibration
#
#   Lower bounds (ramp + margins):
#       1) 1 − α − L_r(f, Q_σ)
#       2) 1 − α − ( L_r(f,P) + L_γ ρ_mix(σ) )
#       3) 1 − α − P( γ_f(X,Y) ≤ L_γ ρ_mix(σ) )
#       4) 1 − α − Q_σ( γ_f(X,Y) ≤ 0 )
# ============================================================

import math
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    import scienceplots
    plt.style.use(["science", "ieee", "no-latex"])
except ImportError:
    plt.style.use("default")

# ---------- Matplotlib style ----------
mpl.rcParams["font.size"] = 14
mpl.rcParams["axes.labelsize"] = 14
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 13
mpl.rcParams["ytick.labelsize"] = 13
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

IEEE_WIDTH_1COL = 3.5   # inches
IEEE_ASPECT     = 0.9
FIGSIZE_1COL    = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

SIGMA_MIN_GRID = 0.0
SIGMA_MAX_GRID = 1.0
N_SIGMA_GRID   = 11
SIGMA_GRID     = np.linspace(SIGMA_MIN_GRID, SIGMA_MAX_GRID, N_SIGMA_GRID)

# ============================================================
# Helper functions
# ============================================================

def coverage_given_threshold(scores: np.ndarray, q: float) -> float:
    """Empirical coverage P(s(X,Y) ≤ q)."""
    scores = np.asarray(scores, dtype=float)
    if scores.size == 0:
        return float("nan")
    return float(np.mean(scores <= q))


@torch.no_grad()
def collect_pseudo_scores_hard(
    loader: DataLoader,
    sigma: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    """
    Hard pseudo-calibration scores on Q_σ:

        ŷ = argmax_k logit_k after T_σ;
        returns scores s(x, ŷ) on 'loader' as a NumPy array.
    """
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(dim=1)

        s = scores_for_labels_from_logits(logits, yhat)
        # Robust: handle tensor or numpy
        if isinstance(s, torch.Tensor):
            s_np = s.detach().cpu().numpy()
        else:
            s_np = np.asarray(s)
        all_scores.append(s_np)

    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])


@torch.no_grad()
def extract_features_batch(x: torch.Tensor) -> torch.Tensor:
    """
    Latent features φ(x).
    Assumes model.features_from_x_nograd(x) exists.
    """
    model.eval()
    return model.features_from_x_nograd(x)


@torch.no_grad()
def estimate_feature_rho_mix(
    loader: DataLoader,
    sigma: float,
    max_batches: int = None,
    trim_quantile: float = 0.9,
):
    """
    Latent-space shift magnitude:

        ρ_mix(σ) ≈ E[ ||φ(X) - φ(T_σ(X))||_2 ],

    using source data (X, Y) ~ P_{XY}.
    Returns (rho_full, rho_trimmed).
    """
    model.eval()
    all_dists = []

    for b, (x, _) in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break

        x = x.to(DEVICE)

        z = extract_features_batch(x)  # φ(X)
        x_shift = apply_Q_transform(x.clone(), sigma)
        z_shift = extract_features_batch(x_shift)  # φ(T_σ(X))

        d = (z - z_shift).norm(p=2, dim=1)  # [B]
        all_dists.append(d.detach().cpu().numpy())

    if not all_dists:
        return float("nan"), float("nan")

    dists = np.concatenate(all_dists, axis=0)
    rho_full = float(dists.mean())

    if trim_quantile is None:
        rho_trim = rho_full
    else:
        q = np.quantile(dists, trim_quantile)
        mask = dists <= q
        rho_trim = float(dists[mask].mean()) if np.any(mask) else rho_full

    return rho_full, rho_trim


def estimate_L_gamma_feature_global() -> float:
    """
    Global Lipschitz constant of the margin in latent space:

        L_γ = sup_{y≠k} ||w_y - w_k||_2

    assuming classifier is linear in φ(x): logits = W φ(x) + b.
    """
    with torch.no_grad():
        W = model.classifier.weight.detach()   # [K, D_latent]
        diff = W.unsqueeze(0) - W.unsqueeze(1)  # [K, K, D]
        norms = diff.norm(p=2, dim=2)          # [K, K]
        Kc = norms.size(0)
        mask = ~torch.eye(Kc, dtype=torch.bool, device=norms.device)
        L_gamma = norms[mask].max().item()
    return float(L_gamma)


@torch.no_grad()
def estimate_ramp_loss_P(loader: DataLoader) -> float:
    """
    Ramp loss on P:

        L_r(f,P) = E[ min{ (1 - γ_f(X,Y))^+, 1 } ].
    """
    model.eval()
    total_ramp = 0.0
    total_n = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)

        hinge = torch.relu(1.0 - margin)  # (1 - γ)^+
        ramp = torch.clamp(hinge, max=1.0)

        bs = x.size(0)
        total_ramp += ramp.sum().item()
        total_n += bs

    return total_ramp / total_n if total_n > 0 else float("nan")


@torch.no_grad()
def estimate_ramp_loss_Q(loader: DataLoader, sigma: float) -> float:
    """
    Ramp loss on Q_σ:

        L_r(f,Q_σ) = E_Q[ min{ (1 - γ_f(X,Y))^+, 1 } ],

    where X is transformed by T_σ.
    """
    model.eval()
    total_ramp = 0.0
    total_n = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)

        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)

        hinge = torch.relu(1.0 - margin)
        ramp = torch.clamp(hinge, max=1.0)

        bs = x.size(0)
        total_ramp += ramp.sum().item()
        total_n += bs

    return total_ramp / total_n if total_n > 0 else float("nan")


@torch.no_grad()
def collect_margins_P(loader: DataLoader) -> np.ndarray:
    """
    Collect margins γ_f(X,Y) on source data (P).
    """
    model.eval()
    all_margins = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        all_margins.append(margin.detach().cpu().numpy())
    return np.concatenate(all_margins, axis=0) if all_margins else np.array([])


@torch.no_grad()
def estimate_margin_tail_Q(
    loader: DataLoader,
    sigma: float,
    threshold: float = 0.0,
) -> float:
    """
    Estimate Q_σ( γ_f(X,Y) ≤ threshold ) using labeled target data
    with shift σ (via T_σ). By default threshold=0.
    """
    model.eval()
    total_count = 0
    total_n = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)

        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)

        mask = (margin <= threshold)
        total_count += mask.sum().item()
        total_n += x.size(0)

    return total_count / total_n if total_n > 0 else float("nan")


# ============================================================
# Main computation
# ============================================================

sigma_vals = SIGMA_GRID.astype(float)
nominal_cov = 1.0 - ALPHA

print("1) Source calibration threshold at σ=0 ...")
scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
q_source = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

print("2) Coverage curves for source, hard pseudo, source-tuned...")

cov_source_Q = []
cov_hard_Q = []

for sigma in sigma_vals:
    sigma_f = float(sigma)

    # True scores on Q_tst under shift σ
    scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma=sigma_f)

    # Source calibration coverage using q_source
    cov_src = coverage_given_threshold(scores_Q_tst_true, q_source)
    cov_source_Q.append(cov_src)

    # Hard pseudo-calibration on Q_σ
    scores_Q_cal_hard = collect_pseudo_scores_hard(
        loader=Q_cal_loader,
        sigma=sigma_f,
        num_classes=NUM_CLASSES,
    )
    q_hard = conformal_quantile(scores_Q_cal_hard, ALPHA, finite_sample=True)
    cov_hard = coverage_given_threshold(scores_Q_tst_true, q_hard)
    cov_hard_Q.append(cov_hard)

cov_source_Q = np.array(cov_source_Q, dtype=np.float64)
cov_hard_Q = np.array(cov_hard_Q, dtype=np.float64)

# Source-tuned pseudo-calibration coverage (Algorithm 1), final epoch
cov_alg_Q = cov_Q_uP_epochs_sigma[-1, :].copy()

print("3) Source ramp loss L_r(f,P) ...")
L_r_source = estimate_ramp_loss_P(P_tst_loader)
print(f"   L_r(f,P) ≈ {L_r_source:.4f}")

print("4) Margins on P ...")
margins_P = collect_margins_P(P_tst_loader)
print(f"   Collected {margins_P.shape[0]} margins on P.")

print("5) Latent ρ_mix(σ) (full & trimmed) ...")
rho_feat_vals = []
rho_feat_trim_vals = []

for sigma in sigma_vals:
    rho_full, rho_trim = estimate_feature_rho_mix(
        loader=P_cal_loader,
        sigma=float(sigma),
        max_batches=None,
        trim_quantile=0.9,
    )
    rho_feat_vals.append(rho_full)
    rho_feat_trim_vals.append(rho_trim)

rho_feat_vals = np.array(rho_feat_vals, dtype=np.float64)
rho_feat_trim_vals = np.array(rho_feat_trim_vals, dtype=np.float64)
print("   ρ_mix full:   ", rho_feat_vals)
print("   ρ_mix trimmed:", rho_feat_trim_vals)

print("6) Global latent L_γ ...")
L_gamma_global = estimate_L_gamma_feature_global()
print(f"   L_γ (latent) ≈ {L_gamma_global:.4f}")

print("7) Ramp loss on Q_σ ...")
L_r_Q_sigmas = []
for sigma in sigma_vals:
    L_r_Q = estimate_ramp_loss_Q(Q_tst_loader, sigma=float(sigma))
    L_r_Q_sigmas.append(L_r_Q)
L_r_Q_sigmas = np.array(L_r_Q_sigmas, dtype=np.float64)
print("   L_r(f,Q_σ):", L_r_Q_sigmas)

print("8) Oracle margin tail Q_σ(γ_f(X,Y) ≤ 0) ...")
margin_tail_Q_sigmas = []
for sigma in sigma_vals:
    tail = estimate_margin_tail_Q(Q_tst_loader, sigma=float(sigma), threshold=0.0)
    margin_tail_Q_sigmas.append(tail)
margin_tail_Q_sigmas = np.array(margin_tail_Q_sigmas, dtype=np.float64)
print("   Q_σ(γ_f(X,Y) ≤ 0):", margin_tail_Q_sigmas)

# ============================================================
# Build ramp-loss-based & margin-based lower bounds
# ============================================================

print("9) Building coverage lower bounds...")

# Use trimmed latent ρ_mix for stability (can switch to rho_feat_vals if desired)
rho_mix_vals = rho_feat_trim_vals

# (1) 1 - α - L_r(f,Q_σ)  (oracle ramp-loss bound on target)
cov_lb_ramp_Q = nominal_cov - L_r_Q_sigmas
cov_lb_ramp_Q = np.clip(cov_lb_ramp_Q, 0.0, 1.0)

# (2) 1 - α - ( L_r(f,P) + L_γ ρ_mix(σ) )  (DA-style ramp bound, no Q labels)
cov_lb_ramp_Prho = nominal_cov - (L_r_source + L_gamma_global * rho_mix_vals)
cov_lb_ramp_Prho = np.clip(cov_lb_ramp_Prho, 0.0, 1.0)

# (3) 1 - α - P( γ_f(X,Y) ≤ L_γ ρ_mix(σ) )  (source-only margin tail)
cov_lb_margin_tail_P = []
for rho in rho_mix_vals:
    theta = L_gamma_global * rho
    if margins_P.size == 0 or not np.isfinite(theta):
        tail_prob = 0.0
    else:
        tail_prob = float(np.mean(margins_P <= theta))
    lb = nominal_cov - tail_prob
    lb = max(0.0, min(1.0, lb))
    cov_lb_margin_tail_P.append(lb)
cov_lb_margin_tail_P = np.array(cov_lb_margin_tail_P, dtype=np.float64)

# (4) 1 - α - Q_σ(γ_f(X,Y) ≤ 0)  (oracle margin-tail bound on target)
cov_lb_margin0_Q = nominal_cov - margin_tail_Q_sigmas
cov_lb_margin0_Q = np.clip(cov_lb_margin0_Q, 0.0, 1.0)

# ============================================================
# Final plot
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE_1COL)

# Empirical coverage
ax.plot(
    sigma_vals,
    cov_source_Q,
    color='r',
    marker="o",
    linestyle="-",
    label="Source Cal.",
)
ax.plot(
    sigma_vals,
    cov_hard_Q,
    color='b',
    marker="s",
    linestyle="-",
    label="Hard Pseudo-Cal.",
)
ax.plot(
    sigma_vals,
    cov_alg_Q,
    color='g',
    marker="^",
    linestyle="-",
    label="Source-tuned Pseudo-Cal.",
)

# Lower bounds

# 1) 1 - α - L_r(f,Q_σ)
ax.plot(
    sigma_vals,
    cov_lb_ramp_Q,
    color='k',
    marker="o",
    linestyle="-.",
    label=r"$1-\alpha - L_r(f,Q_\sigma)$ LB",
)

# # 2) 1 - α - ( L_r(f,P) + L_γ ρ_mix )
# ax.plot(
#     sigma_vals,
#     cov_lb_ramp_Prho,
#     color='orange',
#     marker="o",
#     linestyle="-.",
#     label=r"$1-\alpha - L_r(f,P) - L_\gamma\rho_{\mathrm{mix}}$ LB",
# )

# # 3) 1 - α - P( γ_f(X,Y) ≤ L_γ ρ_mix )
# ax.plot(
#     sigma_vals,
#     cov_lb_margin_tail_P,
#     color='orange',
#     marker="x",
#     linestyle="-.",
#     label=r"$1-\alpha - P(\gamma_f(X,Y) <= L_\gamma\rho_{\mathrm{mix}})$ LB",
# )

# 4) 1 - α - Q_σ(γ_f(X,Y) ≤ 0)
ax.plot(
    sigma_vals,
    cov_lb_margin0_Q,
    color='k',
    marker="x",
    linestyle="-.",
    label=r"$1-\alpha - Q_\sigma(\gamma_f(X,Y) <= 0)$ LB",
)

# Nominal coverage line
ax.axhline(
    nominal_cov,
    linestyle="--",
    linewidth=1.0,
    color="gray",
    label=r"Nominal $1-\alpha$",
)

ax.set_xlabel(r"Shift level $\sigma$")
ax.set_ylabel(r"Coverage on $Q_\sigma$")
ax.set_ylim(0.0, 1.02)
ax.set_xlim(sigma_vals.min(), sigma_vals.max())
ax.grid(True, which="both", linewidth=0.5, alpha=0.5)
ax.legend(frameon=False, loc="lower left")

plt.tight_layout()
plt.savefig("cifar100_coverage_vs_shift_ramp_margin_bounds.pdf", bbox_inches="tight")
plt.savefig("cifar100_coverage_vs_shift_ramp_margin_bounds.png", dpi=300, bbox_inches="tight")
plt.show()

print("Done: plot with ramp-loss & margin-tail lower bounds (including oracle 1-α - Q(γ ≤ 0)).")


In [ ]:
# ============================================================
# Add ECP-style entropy scaling (paper 2406.01416v1) and plot
# alongside your tau-adjusted method.
#
# Assumes these already exist from your notebook:
#   model, DEVICE, NUM_CLASSES, MARGIN, ALPHA, SIGMA_GRID
#   P_cal_loader, P_tst_loader, Q_cal_loader, Q_tst_loader
#   apply_Q_transform
#   predictive_entropy_from_logits
#   samplewise_margin_and_hinge
#   scores_for_labels_from_logits
#   conformal_quantile
#   collect_true_scores
#   coverage_given_threshold
#   expected_set_size_threshold
# ============================================================

import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib as mpl

# --- plotting style (same vibe as your cell) ---
try:
    import scienceplots
    plt.style.use(["science", "ieee", "no-latex"])
except ImportError:
    plt.style.use("default")

mpl.rcParams["font.size"] = 14
mpl.rcParams["axes.labelsize"] = 14
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 13
mpl.rcParams["ytick.labelsize"] = 13
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
plt.rcParams["lines.linewidth"] = 1.0

color_cov_base = "tab:blue"
color_cov_tau  = "tab:orange"
color_cov_ecp  = "tab:green"
color_LB       = "k"

IEEE_WIDTH_1COL = 3.5
IEEE_ASPECT     = 0.9
FIGSIZE_1COL    = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

DATASET_TAG = "cifar100"

nominal_cov = 1.0 - float(ALPHA)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def quantile_higher(a: np.ndarray, q: float) -> float:
    a = np.asarray(a)
    if a.size == 0:
        return float("nan")
    try:
        return float(np.quantile(a, q, method="higher"))
    except TypeError:
        return float(np.quantile(a, q, interpolation="higher"))

@torch.no_grad()
def collect_margins(loader: DataLoader, sigma: float = 0.0) -> np.ndarray:
    """Collect margins γ_f(X,Y) under shift sigma (labels required)."""
    model.eval()
    all_margins = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if float(sigma) != 0.0:
            x = apply_Q_transform(x, float(sigma))
        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        all_margins.append(margin.detach().cpu().numpy())
    return np.concatenate(all_margins, axis=0) if all_margins else np.array([])

def hinge_loss_from_margins(margins: np.ndarray) -> float:
    """L_h = E[(1 - margin)^+]."""
    margins = np.asarray(margins)
    if margins.size == 0:
        return 0.0
    return float(np.mean(np.maximum(1.0 - margins, 0.0)))

@torch.no_grad()
def collect_pseudo_scores_hard(loader: DataLoader, sigma: float) -> np.ndarray:
    """
    Hard pseudo-cal scores:
      yhat = argmax_k logit_k
      score is s(x, yhat) with your score function.
    """
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        if float(sigma) != 0.0:
            x = apply_Q_transform(x, float(sigma))
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(dim=1)
        s = scores_for_labels_from_logits(logits, yhat)  # numpy from your helper
        all_scores.append(np.asarray(s))
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def collect_entropies(loader: DataLoader, sigma: float) -> np.ndarray:
    """Unlabeled predictive entropies h(x) under shift sigma."""
    model.eval()
    ents = []
    for x, _ in loader:
        x = x.to(DEVICE)
        if float(sigma) != 0.0:
            x = apply_Q_transform(x, float(sigma))
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)  # [B]
        ents.append(ent.detach().cpu().numpy())
    return np.concatenate(ents, axis=0) if ents else np.array([])

# ============================================================
# YOUR tau-adjusted method (recomputed here, unchanged logic)
# ============================================================
print("\n=== Tau-adjusted hard pseudo-calibration (your method) ===")

# ---- STEP 1: Delta_P on source P (no clamping) ----
scores_P_cal_pseudo = collect_pseudo_scores_hard(P_cal_loader, sigma=0.0)
q_pseudo_P = conformal_quantile(scores_P_cal_pseudo, ALPHA, finite_sample=True)

scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)
coverage_P_pseudo = coverage_given_threshold(scores_P_tst_true, q_pseudo_P)

Delta_P = float(nominal_cov - coverage_P_pseudo)

margins_P = collect_margins(P_tst_loader, sigma=0.0)
L_h_P = hinge_loss_from_margins(margins_P)

denom_base = float(L_h_P - Delta_P)
target_LB  = float(nominal_cov - L_h_P + Delta_P)

print(f"Nominal coverage (1-α):                 {nominal_cov:.6f}")
print(f"Pseudo-cal threshold on P (q_pseudo_P): {q_pseudo_P:.6f}")
print(f"Coverage on P using q_pseudo_P:         {coverage_P_pseudo:.6f}")
print(f"Delta_P = (1-α) - coverage:             {Delta_P:.6f}")
print(f"Hinge loss L_h(P):                      {L_h_P:.6f}")
print(f"Denominator L_h(P) - Delta_P:           {denom_base:.6f}")
print(f"Target LB = (1-α) - L_h(P) + Delta_P:   {target_LB:.6f}")

# ---- STEP 2: sweep sigma and compute tau(sigma) + coverage/ESS ----
sigma_vals = np.asarray(SIGMA_GRID, dtype=float)
N_SIGMA = len(sigma_vals)

coverage_hard     = np.zeros(N_SIGMA, dtype=np.float64)
coverage_hard_tau = np.zeros(N_SIGMA, dtype=np.float64)
ESS_hard          = np.zeros(N_SIGMA, dtype=np.float64)
ESS_hard_tau      = np.zeros(N_SIGMA, dtype=np.float64)

tau_by_sigma       = np.zeros(N_SIGMA, dtype=np.float64)
LB_empirical_sigma = np.zeros(N_SIGMA, dtype=np.float64)
hinge_loss_Q_sigma = np.zeros(N_SIGMA, dtype=np.float64)

TAU_MIN = 0.0
TAU_MAX = None
EPS = 1e-12

print("\n=== Sweep sigma: compute q_hard(σ), tau(σ), coverage/ESS ===")
for i, sigma in enumerate(sigma_vals):
    sigma_f = float(sigma)

    # Hard pseudo-calibrate on Q_cal (shifted)
    scores_Q_cal_hard = collect_pseudo_scores_hard(Q_cal_loader, sigma=sigma_f)
    q_hard = conformal_quantile(scores_Q_cal_hard, ALPHA, finite_sample=True)

    # Evaluate on Q_tst with true labels
    scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma=sigma_f)
    cov_base = coverage_given_threshold(scores_Q_tst_true, q_hard)
    ESS_base = expected_set_size_threshold(Q_tst_loader, sigma_f, q_hard)

    coverage_hard[i] = cov_base
    ESS_hard[i]      = ESS_base

    # Hinge loss on Q_tst (true labels)
    margins_Q = collect_margins(Q_tst_loader, sigma=sigma_f)
    L_h_Q = hinge_loss_from_margins(margins_Q)
    hinge_loss_Q_sigma[i] = L_h_Q

    # Solve tau
    if abs(denom_base) <= EPS:
        tau_raw = 0.0
        if L_h_Q > EPS:
            print(f"  WARNING σ={sigma_f:.3f}: L_h(P)-Δ_P ~ 0 but L_h(Qσ)>0; equation not solvable with finite τ. Using τ=0.")
    else:
        tau_raw = 2.0 * (L_h_Q / denom_base - 1.0)

    tau = float(max(TAU_MIN, tau_raw))
    if TAU_MAX is not None:
        tau = float(np.clip(tau, TAU_MIN, TAU_MAX))

    tau_by_sigma[i] = tau

    denom = 1.0 + tau / 2.0
    LB_emp = nominal_cov - (L_h_Q / denom) if denom > 0 else float("nan")
    LB_empirical_sigma[i] = float(LB_emp)

    # Apply adjusted threshold and evaluate
    q_hard_tau = q_hard + tau
    cov_tau = coverage_given_threshold(scores_Q_tst_true, q_hard_tau)
    ESS_tau = expected_set_size_threshold(Q_tst_loader, sigma_f, q_hard_tau)

    coverage_hard_tau[i] = cov_tau
    ESS_hard_tau[i]      = ESS_tau

# ============================================================
# PAPER method: ECP-style entropy scaling
#
# Your score is "badness" s(x,y)=max_other - logit_y, and sets use s<=q.
# Paper scales "goodness" scores by a factor >=1 and keeps τ fixed.
# To make sets larger with higher entropy in *your* direction, we do:
#
#   q_P_fixed = conformal quantile from SOURCE P (fixed across σ)
#   u_Qσ      = (1-α)-quantile of entropies on unlabeled Qσ test
#   wσ        = max(1, u_Qσ)
#   Effective rule: s(x,y) / wσ <= q_P_fixed   <=>  s(x,y) <= q_P_fixed * wσ
#
# So quantile is fixed from P; along σ, only the entropy-derived scale changes.
# ============================================================
print("\n=== ECP-style entropy scaling (paper) ===")

# Source calibration quantile (TRUE labels on P_cal) — fixed across all σ
scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
q_P_fixed = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)
print(f"Fixed source quantile q_P_fixed (from P_cal true labels): {q_P_fixed:.6f}")

coverage_ecp = np.zeros(N_SIGMA, dtype=np.float64)
ESS_ecp      = np.zeros(N_SIGMA, dtype=np.float64)
u_ecp_sigma  = np.zeros(N_SIGMA, dtype=np.float64)
w_ecp_sigma  = np.zeros(N_SIGMA, dtype=np.float64)
q_ecp_sigma  = np.zeros(N_SIGMA, dtype=np.float64)

for i, sigma in enumerate(sigma_vals):
    sigma_f = float(sigma)

    # Unlabeled entropy quantile on Qσ test set
    ent_Q = collect_entropies(Q_tst_loader, sigma=sigma_f)
    u = quantile_higher(ent_Q, 1.0 - float(ALPHA))
    w = float(max(1.0, u))

    u_ecp_sigma[i] = float(u)
    w_ecp_sigma[i] = float(w)

    # Effective threshold after scaling (see note above)
    q_eff = float(q_P_fixed * w)
    q_ecp_sigma[i] = q_eff

    # Evaluate on Qσ test with true labels
    scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma=sigma_f)
    coverage_ecp[i] = coverage_given_threshold(scores_Q_tst_true, q_eff)
    ESS_ecp[i]      = expected_set_size_threshold(Q_tst_loader, sigma_f, q_eff)

print("Done. (If you want, inspect u_ecp_sigma / w_ecp_sigma / q_ecp_sigma.)")

# ============================================================
# Plot: coverage + expected set size vs sigma (tau vs ECP)
# ============================================================
fig, ax1 = plt.subplots(figsize=FIGSIZE_1COL)
ax2 = ax1.twinx()

# Coverage (left y-axis)
ax1.plot(sigma_vals, coverage_hard,      "-",  color=color_cov_base, label=r"Coverage with $q_{\tilde{Q},\alpha}$")
ax1.plot(sigma_vals, coverage_hard_tau,  "-",  color=color_cov_tau,  label=r"Coverage with $q_{\tilde{Q},\alpha}+\tau(\sigma)$")
ax1.plot(sigma_vals, coverage_ecp,       "-",  color=color_cov_ecp,  label=r"Coverage with ECP entropy scaling (fixed $q_P$)")
ax1.plot(sigma_vals, LB_empirical_sigma, "-.", color=color_LB,       label=r"Hinge-loss LB of $q_{\tilde{Q},\alpha}+\tau(\sigma)$")
ax1.axhline(y=nominal_cov, linestyle=":", color="0.5", alpha=0.9, label=r"Nominal $1-\alpha$")

# ESS (right y-axis)
ax2.plot(sigma_vals, ESS_hard,     "--", color=color_cov_base, label=r"ESS with $q_{\tilde{Q},\alpha}$")
ax2.plot(sigma_vals, ESS_hard_tau, "--", color=color_cov_tau,  label=r"ESS with $q_{\tilde{Q},\alpha}+\tau(\sigma)$")
ax2.plot(sigma_vals, ESS_ecp,      "--", color=color_cov_ecp,  label=r"ESS with ECP entropy scaling (fixed $q_P$)")

ax1.set_xlabel(r"Shift level $\sigma$")
ax1.set_ylabel(r"Coverage on $Q_\sigma$")
ax1.set_ylim(0.0, 1.02)
ax1.set_xlim(float(sigma_vals.min()), float(sigma_vals.max()))
ax1.grid(True, which="both", linewidth=0.5, alpha=0.5)

ax2.set_ylabel(r"Expected set size on $Q_\sigma$")
ax2.set_ylim(0.0, NUM_CLASSES + 0.5)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc=(0.01, 0.05))

plt.tight_layout()
out_base = f"{DATASET_TAG}_coverage_ess_vs_shift_tau_vs_ecp_entropy_scaling"
plt.savefig(out_base + ".pdf")
plt.savefig(out_base + ".png", dpi=300)
plt.show()

print(f"\nSaved plot: {out_base}.[pdf|png]")
print("\nQuick sanity:")
for i in [0, min(5, N_SIGMA-1), N_SIGMA-1]:
    print(f"  σ={sigma_vals[i]:.2f} | ECP u={u_ecp_sigma[i]:.3f} w={w_ecp_sigma[i]:.3f} q_eff={q_ecp_sigma[i]:.3f} | "
          f"cov_ecp={coverage_ecp[i]:.3f} ess_ecp={ESS_ecp[i]:.2f} | "
          f"cov_tau={coverage_hard_tau[i]:.3f} ess_tau={ESS_hard_tau[i]:.2f}")


In [ ]:
save_path = "cifar100_linear_uP_model.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved model weights to {save_path}")